# 🕸️ Module 1.2 — Multi-Agent Orchestration

**Domain 1 · Agentic Architecture & Orchestration** (27% of the exam)
**Task 1.2 · Multi-Agent Orchestration** · ⏱️ ~60 minutes
**Source:** [claudecertificationguide.com/learn/1-agentic-architecture/1-2-orchestration-patterns](https://claudecertificationguide.com/learn/1-agentic-architecture/1-2-orchestration-patterns)

Welcome back! Module 1.1 was about *one* agent looping on its own. This one is
about several agents working together without stepping on each other — and,
just as importantly, about a very specific way that setup goes wrong.

### 🎯 What you'll build

A working **hub-and-spoke research coordinator**: it breaks a broad topic into
subtopics, spawns isolated subagents that share no memory with anything else,
combines their work, checks its own coverage, and re-delegates until nothing's
missing. Then you'll deliberately break it the exact way the exam describes,
and watch the failure happen for real.

### ✅ What you'll walk away knowing

1. The hub-and-spoke pattern — why *all* communication routes through one coordinator
2. The isolation principle — subagents share no memory, context, or state, ever
3. Why coverage gaps are (almost always) a decomposition bug, not a subagent bug
4. The coordinator's four core jobs, including the iterative refinement loop
5. How to diagnose a multi-agent failure by tracing it back to its origin

---

> **💳 Heads up — this notebook makes more real, billed API calls than Module
> 1.1 did (roughly 15–20 short ones for a full run top to bottom).** That's
> not an accident — fanning out to multiple subagents is the whole point of
> this module, and the cost of that fan-out is itself worth noticing. Still
> just a few cents on Claude Sonnet 5, and every call is short (a few hundred
> tokens). Read first and run later if you'd rather not spend anything yet.

## 🔧 Setup

Same as Module 1.1 — if you already have `anthropic` installed and
`ANTHROPIC_API_KEY` set, skip straight to the cell below.

```bash
pip install anthropic
```

```bash
# Windows (PowerShell)
$env:ANTHROPIC_API_KEY = "sk-ant-..."

# macOS / Linux (bash/zsh)
export ANTHROPIC_API_KEY="sk-ant-..."
```


In [ ]:
import os
from anthropic import Anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. Set it in your shell, then restart the "
        "kernel and run this cell again -- see the Setup section above."
    )

client = Anthropic()
MODEL = "claude-sonnet-5"

print("Connected. Using model:", MODEL)


## 🔑 Key Concept: Hub-and-Spoke Architecture

The standardized multi-agent pattern this exam tests: two roles, one
centralized, hierarchical structure.

| Role | Responsibilities | Position |
|---|---|---|
| **Coordinator** | Receives the task, decomposes it, selects subagents, passes context, aggregates results, handles errors, routes all information | Center (hub) |
| **Subagents** | Execute one specialized task (search, analysis, synthesis, ...), report back to the coordinator | Edges (spokes) |

### The cardinal rule

> "ALL communication flows through the coordinator. Subagents never
> communicate directly with each other. Never. Not for efficiency, not for
> convenience, not for any reason."

**Why this rule exists (three exam-tested benefits):**

1. **Observability** — every message is loggable in one place
2. **Consistent error handling** — one uniform recovery policy, not N different ones
3. **Controlled information flow** — the coordinator decides exactly what each subagent sees

> A quick aside: real Claude Code does allow nested parent→child subagent
> delegation. The exam still tests the strict hub-and-spoke model — no direct
> subagent-to-subagent communication — as the correct answer, so that's what
> we build here.


## 🔑 Key Concept: The Isolation Principle

The single most-tested misunderstanding in this module: **subagents do not
automatically inherit the coordinator's context, history, or memory.**

**What a subagent does *not* have, unless explicitly given it:**
- The coordinator's system prompt
- Any previous messages from the coordinator's conversation
- Results from any other subagent
- Access to any "shared memory" or global state

**Across invocations, too:** call the same subagent twice, and the second
call knows *nothing* about the first. Every invocation starts from zero.

**What this means for you as the coordinator author:** you must be
deliberate and explicit. If a synthesis subagent needs a search subagent's
results, you type those results directly into the synthesis subagent's
prompt — there is no shared store for it to "check".


## 🔑 Key Concept: The Coordinator's Four Jobs

**1. Dynamic subagent selection** — decide *which* subagents a query actually
needs. "What's the capital of France?" needs one search call, not a full
research→analysis→synthesis pipeline. Routing everything through every
subagent isn't thoroughness, it's waste.

**2. Research scope partitioning** — when multiple subagents are used, give
each a distinct slice (different subtopics, different source types) so they
don't duplicate each other's work.

**3. Iterative refinement** — evaluate the aggregated output for gaps and, if
incomplete, re-delegate with *targeted* follow-up queries. This is a loop,
not a single shot.

**4. Centralized communication routing** — every message, still, always,
through the coordinator. This is the same cardinal rule from above, restated
as an ongoing responsibility rather than a one-time design choice.


## 🔑 Key Concept: The Narrow Decomposition Failure

This is *the* exam-tested failure pattern for this module, so it's worth
sitting with before writing any code.

**The scenario:** a coordinator asked to research "the impact of AI on
creative industries" decomposes it into *only* visual-arts subtopics —
music, writing, and film never get assigned to anyone.

**Root cause:** the coordinator's decomposition. Not any subagent.

- The search subagent searched thoroughly for what it was assigned.
- The synthesis subagent synthesized everything it received, accurately.
- But music/writing/film were never assigned to anyone — so nobody ever
  looked.

> "Trace failures to their origin. When a multi-agent system produces a
> report that misses entire categories, do not blame the subagents — check
> the coordinator's decomposition."

**A distinction worth keeping:**

| Gap type | Usual cause |
|---|---|
| **Scope gap** (a whole category is just missing) | Almost always the coordinator's decomposition |
| **Depth gap** (a category is covered, but shallowly) | May genuinely involve a downstream subagent |

### Practical example we'll build against

A research system asked about "renewable energy technologies" decomposes it
into only **solar** and **wind**. The report is thorough on both — and silent
on geothermal, tidal, biomass, and nuclear fusion.

| What went right | Why it doesn't matter |
|---|---|
| The search subagent returned solid results for every query it got | It was only ever asked about solar and wind |
| The synthesis subagent combined everything it received, accurately | It only ever received solar and wind research |

**Wrong fixes:** better search queries, a more capable synthesis model, more
subagents. **None of these touch the actual bug** — a coordinator that only
ever assigns two categories. **Correct fix:** widen the decomposition.

This exact scenario is what Tasks 2–6 below are built around.


## 🛠️ Build Exercise — Task 1: Create a Coordinator Shell

**Objective:** the central hub that will accept a topic and return a
structured report — with no subagent calls yet. Just the shape.

**Why this matters:** the coordinator owns decomposition, subagent selection,
and aggregation. Before writing any of that logic, it's worth being explicit
about *what a coordinator is* — a system prompt defining the orchestrating
role, plus a report structure everything else will fill in.


In [ ]:
COORDINATOR_SYSTEM_PROMPT = (
    "You are the coordinator agent in a hub-and-spoke multi-agent research "
    "system. You decompose broad topics into distinct subtopics, decide "
    "which subagents to invoke, pass each subagent exactly the context it "
    "needs, and aggregate their results into one coherent report. Subagents "
    "never talk to each other -- every piece of information flows through you."
)


def new_report(topic: str) -> dict:
    """The report 'shape' the coordinator fills in as it works.

    Task 1 only builds this shell -- no subagents invoked yet, no
    decomposition yet. Tasks 2-6 progressively fill each field in.
    """
    return {
        "topic": topic,
        "assigned_subtopics": [],
        "sections": {},     # subtopic -> subagent research text
        "coverage": None,   # filled in once we can assess it (Task 4)
        "iterations": 0,
    }


report = new_report("renewable energy technologies")
print(report)


## 🛠️ Build Exercise — Task 2: Task Decomposition Logic

**Objective:** break a topic into at least 5 distinct, non-overlapping
subtopics covering its full breadth.

**Why this matters:** this is the exact spot the exam's failure pattern lives
in. Below, `decompose_topic_antipattern` is the bug itself, written out as
real code — a coordinator whose decomposition is hard-coded and narrow no
matter what topic comes in. `decompose_topic` is the fix: a real,
breadth-guided call to Claude. We keep the antipattern **active** (not
commented out, unlike the anti-patterns further down) because we reuse it
deliberately in Tasks 4 and 5, and again in the case-study walkthrough near
the end.


In [ ]:
import json as _json
import re as _re


def decompose_topic_antipattern(topic: str) -> list:
    """DO NOT USE -- kept ACTIVE on purpose. This is the exact case-study
    bug: a coordinator whose decomposition logic is hard-coded and narrow,
    completely ignoring whatever topic actually comes in.
    """
    return ["solar power", "wind power"]


def decompose_topic(topic: str) -> list:
    """The fix -- decompose via a real, breadth-guided Claude call instead
    of a hard-coded, narrow list.
    """
    system_prompt = (
        "You are the coordinator agent in a hub-and-spoke multi-agent "
        "research system. When given a topic, decompose it into distinct, "
        "non-overlapping subtopics that together cover its FULL breadth -- "
        "at least 5, ideally 6 or more for a broad topic. Respond with ONLY "
        "a JSON array of short subtopic strings and nothing else -- no "
        "markdown, no commentary."
    )
    user_prompt = f"Decompose this research topic for full-breadth coverage: {topic!r}"
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )
    text = "".join(b.text for b in response.content if b.type == "text").strip()

    try:
        subtopics = _json.loads(text)
    except ValueError:
        # Real models don't always follow formatting instructions perfectly --
        # fall back to a forgiving line/bullet parser instead of crashing.
        subtopics = [
            _re.sub(r'^[-*\d.)\s"]+|["]+$', "", line).strip()
            for line in text.splitlines() if line.strip()
        ]
    return [s for s in subtopics if s]


topic = "renewable energy technologies"
print("Antipattern decomposition:", decompose_topic_antipattern(topic))
print("Guided decomposition:     ", decompose_topic(topic))


## 🛠️ Build Exercise — Task 3: Spawn Two Subagents, Explicitly Passing Context

**Objective:** a web-search subagent and a document-analysis subagent, each
given *everything* it needs directly in its prompt — nothing implicit.

**Why this matters:** this is the isolation principle, in code. The
document-analysis subagent below cannot "ask" the web-search subagent for its
findings, and it has no memory of anything the coordinator has done. Every
fact it uses has to be typed directly into its prompt, by us, right now.

This cell makes 2 real API calls, on a single subtopic (`"solar power"`) —
full production code would repeat this pattern for every assigned subtopic.


In [ ]:
def run_subagent(system_prompt: str, user_prompt: str) -> str:
    """One fresh, isolated API call -- a brand-new `messages` list every
    single time. This is what "subagents share no memory" looks like in
    code: there is no object here that could carry state between calls even
    if we wanted it to.
    """
    response = client.messages.create(
        model=MODEL,
        max_tokens=400,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
    )
    return "".join(b.text for b in response.content if b.type == "text")


WEB_SEARCH_SYSTEM_PROMPT = (
    "You are a web-search research subagent in a multi-agent system. You "
    "have no memory of any other conversation or subagent -- you only know "
    "what is in the prompt you were given. Given a subtopic and a research "
    "goal, produce a concise, factual research brief (3-5 sentences) as if "
    "you had searched the web for it."
)

DOCUMENT_ANALYSIS_SYSTEM_PROMPT = (
    "You are a document-analysis research subagent in a multi-agent system. "
    "You have no memory of any other conversation or subagent -- you only "
    "know what is in the prompt you were given, including any prior "
    "research explicitly provided to you there. Synthesize a short "
    "analytical summary (3-5 sentences) highlighting key trends and "
    "challenges."
)

subtopic = "solar power"

search_prompt = f"Research subtopic: {subtopic}\nOverall research goal: {topic}"
search_brief = run_subagent(WEB_SEARCH_SYSTEM_PROMPT, search_prompt)
print("Web-search subagent output:")
print(search_brief)

# The document-analysis subagent gets the search results EXPLICITLY typed
# into its prompt. It cannot fetch them itself -- there is nowhere for it to
# fetch them FROM.
analysis_prompt = (
    f"Research subtopic: {subtopic}\n"
    f"Overall research goal: {topic}\n"
    f"Prior research findings (from the web-search subagent):\n{search_brief}\n\n"
    "Synthesize an analytical summary from the above findings."
)
analysis = run_subagent(DOCUMENT_ANALYSIS_SYSTEM_PROMPT, analysis_prompt)
print()
print("Document-analysis subagent output:")
print(analysis)


Notice what just happened: we never passed the coordinator's own
conversation to either subagent, and we never let the document-analysis
subagent go fetch the web-search subagent's output on its own. We typed it
directly into its prompt, ourselves, as the coordinator. That's the isolation
principle — not a warning in a docstring, but the actual shape of the code.


## 🛠️ Build Exercise — Task 4: Aggregate Results and Evaluate Coverage

**Objective:** combine subagent results and assess whether they cover the
full breadth of the topic — structured, not narrative.

**Why this matters:** this is where "we have thorough coverage of these
topics" and "we are completely missing these other topics" get told apart.
Notice that `assess_coverage` below needs **zero API calls** — it's a
deterministic check against what the coordinator *assigned*, which is
exactly the point: completeness is decided at decomposition time, not at
research time.


In [ ]:
REQUIRED_CATEGORIES = ["solar", "wind", "geothermal", "tidal", "biomass", "fusion"]


def assess_coverage(assigned_subtopics: list) -> dict:
    """Pure, deterministic coverage check -- no LLM call needed.

    Checks which REQUIRED_CATEGORIES are represented by what the coordinator
    actually ASSIGNED, not by how good the resulting research text is. This
    mirrors the module's point exactly: completeness is a decomposition
    property, not a subagent-quality property.
    """
    covered = sorted({
        category for category in REQUIRED_CATEGORIES
        if any(category in s.lower() for s in assigned_subtopics)
    })
    missing = [c for c in REQUIRED_CATEGORIES if c not in covered]
    completeness_pct = round(100 * len(covered) / len(REQUIRED_CATEGORIES))
    return {"covered": covered, "missing": missing, "completeness_pct": completeness_pct}


# Demonstrate immediately against the case-study bug's assignment. No
# research needed yet -- this is purely about what got ASSIGNED.
narrow_assignment = decompose_topic_antipattern(topic)
print("Narrow decomposition assigned:", narrow_assignment)
print("Coverage assessment:          ", assess_coverage(narrow_assignment))


## 🛠️ Build Exercise — Task 5: Iterative Refinement Loop

**Objective:** detect coverage gaps and re-delegate to subagents with
targeted queries until coverage is sufficient (or a cap is hit).

**Why this matters:** this is what separates a real coordinator from a
dumb dispatcher. Single-shot delegation isn't enough — a coordinator has to
check its own work and go back for what's missing.

> One combined "research subagent" per subtopic is used below, rather than
> Task 3's two-role (search → analysis) pattern, purely to keep a full
> multi-subtopic run affordable. The orchestration code around it — decompose,
> assign, check coverage, re-delegate — is identical either way; you'd simply
> call two subagents instead of one inside `research_subtopic` for full fidelity.


In [ ]:
def research_subtopic(subtopic: str, topic_: str) -> str:
    """One isolated research-subagent call for a single subtopic."""
    system_prompt = (
        "You are a research subagent in a multi-agent system. You have no "
        "memory of any other conversation or subagent. Given a subtopic and "
        "a research goal, produce a concise research brief (3-5 sentences) "
        "covering key facts, trends, and current challenges."
    )
    user_prompt = f"Research subtopic: {subtopic}\nOverall research goal: {topic_}"
    return run_subagent(system_prompt, user_prompt)


def run_coordinator(topic_: str, decompose_fn, max_iterations: int = 3):
    """The full hub-and-spoke coordinator loop:

      decompose -> research each newly assigned subtopic -> assess coverage
      -> if gaps remain, add targeted subtopics for exactly the missing
      categories and go around again -> stop once covered, or once
      max_iterations is hit (the safety net, same role as Module 1.1's
      MAX_ITERATIONS -- not the primary mechanism, just the backstop).
    """
    assigned_subtopics = decompose_fn(topic_)
    results = {}
    iteration = 0

    while True:
        iteration += 1

        new_subtopics = [s for s in assigned_subtopics if s not in results]
        for subtopic_ in new_subtopics:
            results[subtopic_] = research_subtopic(subtopic_, topic_)

        coverage = assess_coverage(assigned_subtopics)
        print(f"[iteration {iteration}] assigned={assigned_subtopics}")
        print(f"    completeness={coverage['completeness_pct']}% missing={coverage['missing']}")

        if not coverage["missing"] or iteration >= max_iterations:
            return assigned_subtopics, results, coverage, iteration

        # Targeted re-delegation: assign exactly the missing categories,
        # then loop back around to research just those.
        assigned_subtopics = assigned_subtopics + coverage["missing"]


### Proving the loop actually loops

The cell above only *defines* the machinery. Let's prove the safety net
really works, on purpose, using the deliberately-bad decomposition from
Task 2 — with refinement turned ON this time (unlike the case-study
walkthrough near the end of this notebook, which runs the same bad
decomposition with NO refinement, to show the raw failure).

This makes about 6 real API calls (2 subtopics, then 4 more once the gaps
are found).


In [ ]:
assigned, results, coverage, iterations_used = run_coordinator(
    topic, decompose_topic_antipattern, max_iterations=3
)

print()
print("Final assigned subtopics:", assigned)
print("Final coverage:          ", coverage)
print("Iterations used:         ", iterations_used)

assert coverage["missing"] == [], "expected the refinement loop to close every gap"
print()
print("The safety net worked: a narrow starting decomposition still reached full")
print("coverage, because the coordinator kept checking its own work and asking")
print("for exactly what was missing -- not because the first guess was good.")


## 🛠️ Build Exercise — Task 6: Test with Renewable Energy, Verify Full Coverage

**Objective:** run the complete system on "renewable energy technologies" and
confirm the final report covers all six categories: solar, wind, geothermal,
tidal, biomass, fusion.

**Why this matters:** this is the module's exact test case. If your run only
covers solar and wind, the root cause is the decomposition — the diagnostic
the exam wants you to make. This time we hand the coordinator the *good*
decomposition function from Task 2, so we get to see what a well-guided
decomposition looks like when it's allowed to do its job.


In [ ]:
assigned, results, coverage, iterations_used = run_coordinator(
    topic, decompose_topic, max_iterations=3
)

print()
print("Final assigned subtopics:", assigned)
print("Final coverage:          ", coverage)
print("Iterations used:         ", iterations_used)
print()

if coverage["missing"]:
    print("Even the guided decomposition missed a category or two on this run --")
    print("that's fine, and exactly what the Task 5 refinement loop is FOR. Real")
    print("model output varies call to call; the safety net (not the prompt) is")
    print("what actually guarantees the outcome.")
else:
    print("Full coverage on the very first pass -- a well-guided decomposition")
    print("needed zero refinement iterations to get there.")

# Assemble the final report using Task 1's shell.
report = new_report(topic)
report["assigned_subtopics"] = assigned
report["sections"] = results
report["coverage"] = coverage
report["iterations"] = iterations_used

print()
print(f"=== {topic.title()} -- Final Report ===")
for subtopic_, text in report["sections"].items():
    print()
    print(f"## {subtopic_}")
    print(text)
print()
print(f"Coverage: {report['coverage']['completeness_pct']}% "
      f"({len(report['coverage']['covered'])}/{len(REQUIRED_CATEGORIES)} categories)")
print(f"Refinement iterations used: {report['iterations']}")


## ⚠️ Four Anti-Patterns to Avoid

| # | Anti-pattern | Why it fails | Fix |
|---|---|---|---|
| 1 | Blaming downstream subagents for coverage gaps | Subagents research what they're assigned — no more, no less | Trace failures to the coordinator's decomposition first |
| 2 | Assuming subagents share memory or inherit context | Every invocation is completely isolated, always | Pass everything a subagent needs directly in its prompt |
| 3 | Direct inter-subagent communication (even "just for efficiency") | Breaks observability, consistent error handling, and controlled information flow | Route everything through the coordinator, no exceptions |
| 4 | Adding more subagents to fix a decomposition problem | New subagents get equally narrow assignments from the same narrow decomposition logic | Fix the decomposition, not the headcount |

Each is written below as real, working code — then **commented out** so you
can see the shape of the mistake without ever running it. The exception is
the case study that follows the four traps, where we run the actual bug for
real, on purpose, so you can watch it happen.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 1 -- Blaming downstream subagents for a coverage gap
# ============================================================
# Commented out on purpose.
#
# def diagnose_coverage_gap_antipattern(report_sections: dict) -> str:
#     return ("Fix: use better search queries in the web-search subagent, "
#             "or a more capable synthesis model.")
#
# Why it fails: this never once looks at what the coordinator actually
# ASSIGNED. If assigned_subtopics was only ["solar power", "wind power"], no
# amount of query-tuning or model-upgrading can make geothermal, tidal,
# biomass, or fusion appear -- nobody was ever asked to look for them.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 2 -- Assuming subagents share memory or inherit context
# ============================================================
# Commented out on purpose.
#
# shared_state = {}   # <-- the mistake starts right here
#
# def run_subagent_antipattern_2(role_prompt, subtopic):
#     # Assumes the subagent can just "know" prior results via shared_state,
#     # instead of receiving them explicitly in its own prompt.
#     response = client.messages.create(
#         model=MODEL, max_tokens=400, system=role_prompt,
#         messages=[{"role": "user", "content": f"Research {subtopic}. "
#                    f"Consider prior findings: {shared_state.get('prior')}"}],
#     )
#     return response
#
# Why it fails: nothing about the real Messages API gives a subagent access
# to shared_state, or to any other subagent's conversation. If the prior
# findings you need aren't typed directly into THIS prompt, they don't exist
# as far as this call is concerned -- shared_state.get("prior") might be
# stale, empty, or simply the wrong subtopic's data, and there's no way to
# know from inside the call.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 3 -- Direct inter-subagent communication
# ============================================================
# Commented out on purpose.
#
# def document_analysis_subagent_antipattern_3(subtopic):
#     # "More efficient" -- let this subagent call the web-search subagent
#     # itself, instead of waiting for the coordinator to pass results along.
#     search_result = web_search_subagent_function(subtopic)   # <-- the mistake
#     return run_subagent(DOCUMENT_ANALYSIS_SYSTEM_PROMPT,
#                          f"Analyze: {search_result}")
#
# Why it fails: even if it "works" today, it breaks all three benefits of
# hub-and-spoke at once -- the coordinator can no longer observe this call in
# its own logs, can't apply its own error-handling policy to it, and no
# longer controls what information reached this subagent. "More efficient"
# is not on the list of acceptable reasons -- the module says so directly.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 4 -- Adding more subagents to fix a decomposition problem
# ============================================================
# Commented out on purpose.
#
# def run_coordinator_antipattern_4(topic_):
#     assigned = decompose_topic_antipattern(topic_)   # still just 2 categories
#     # "Let's throw more subagents at it" -- three search agents and three
#     # analysis agents per subtopic, instead of fixing the decomposition:
#     results = {}
#     for subtopic_ in assigned:
#         for _ in range(3):
#             results[subtopic_] = research_subtopic(subtopic_, topic_)
#     return results
#
# Why it fails: tripling the subagents still only researches ["solar power",
# "wind power"] -- three times as thoroughly, at three times the cost, with
# geothermal/tidal/biomass/fusion exactly as missing as before. More
# subagents cannot widen an assignment list they were never given a say in.


## 📖 Case Study, Live: The Renewable Energy Coverage Gap

Everything above was described. This is the module's exact scenario, run for
real, single-shot — **no refinement loop this time** — so you see the raw
failure the way a production system without a safety net would produce it.

This cell makes 2 real API calls.


In [ ]:
bad_assignment = decompose_topic_antipattern(topic)
narrow_results = {s: research_subtopic(s, topic) for s in bad_assignment}

print("=== What the coordinator assigned ===")
print(bad_assignment)
print()
print("=== What each subagent produced (both did a genuinely good job) ===")
for subtopic_, text in narrow_results.items():
    print(f"\n## {subtopic_}\n{text}")

narrow_coverage = assess_coverage(bad_assignment)
print()
print("=== Coverage assessment ===")
print(narrow_coverage)


def diagnose_coverage_gap(assigned_subtopics: list, coverage: dict) -> str:
    """The correct diagnostic -- check the coordinator's decomposition FIRST,
    before ever looking at subagent output quality."""
    if coverage["missing"]:
        return (
            f"Root cause: the coordinator's decomposition never assigned "
            f"{coverage['missing']}. Every subagent that DID run produced solid "
            f"work -- the fix is to widen decompose_topic, not to touch the "
            f"subagents at all."
        )
    return "Decomposition covers all required categories."


print()
print("=== Correct diagnosis ===")
print(diagnose_coverage_gap(bad_assignment, narrow_coverage))


## 🎓 Practice Scenario (from the module)

> A multi-agent research system produces a report on "renewable energy
> technologies" covering only solar and wind power. Each subagent produced
> thorough, well-sourced coverage of its assigned topic. The web-search
> subagent returned relevant results for every query it received. The
> synthesis subagent accurately combined all research it was given. What is
> the most likely root cause of the coverage gap?
>
> - A. The document-analysis subagent had no access to sources on the other categories
> - B. The synthesis subagent failed to identify gaps and request more coverage
> - C. The web-search subagent used queries that were too narrow
> - D. The coordinator's decomposition never assigned the missing categories to any subagent
>
> **Answer: D.** A, B, and C all blame a subagent for not doing something it
> was never asked to do — exactly the anti-pattern this module warns about.
> Every subagent performed correctly on what it was actually given.


## 🏆 Key Takeaways for Exam Prep

1. **Hub-and-spoke is non-negotiable** — the coordinator is the center; all
   communication flows through it. Direct inter-subagent communication is
   always wrong on this exam.
2. **Isolation is absolute** — subagents inherit no context, no memory, no
   shared state. The coordinator must explicitly pass everything.
3. **Decomposition determines completeness** — scope gaps (whole categories
   missing) are almost always a coordinator decomposition bug, not a
   subagent performance problem.
4. **Diagnose from the origin** — when output is incomplete, check the
   coordinator's decomposition before blaming any subagent.
5. **Iterative refinement is core** — evaluate, find gaps, re-delegate with
   targeted queries, re-evaluate. Single-shot delegation isn't enough.
6. **Observability, consistent error handling, controlled information flow**
   — these three benefits are *why* hub-and-spoke wins, even when direct
   communication looks more efficient on paper.


---

## 🎉 Quick-Fire Recap — See If It Stuck

You built a coordinator, watched two isolated subagents hand work to each
other only through explicit prompts, broke the system exactly the way the
exam describes, and then watched a refinement loop heal a *different* run of
that same break. Try these from memory before scrolling back up.

**1. In one sentence: what makes hub-and-spoke "hub-and-spoke"?**
> 💡 One coordinator at the center through which *every* message flows;
> subagents at the edges that never talk to each other, ever, for any reason.

**2. Your document-analysis subagent needs the web-search subagent's
findings. How does it get them?**
> 💡 You — the coordinator — type them directly into its prompt. There's no
> shared store, no memory it can check on its own; if it's not in the prompt,
> it doesn't exist for that call.

**3. A report on "renewable energy" only covers solar and wind, and every
subagent did solid work on what it received. Who's at fault?**
> 💡 The coordinator's decomposition. It never assigned geothermal, tidal,
> biomass, or fusion to anyone — no subagent can research a topic it was
> never told about.

**4. Someone proposes letting two subagents talk to each other directly, "just
this once, for efficiency." What do you say?**
> 💡 No — that quietly breaks observability, consistent error handling, and
> controlled information flow all at once. "More efficient" isn't on the
> list of acceptable exceptions; the module says so in as many words.

**5. Your coordinator's report is missing categories. A teammate suggests
adding three more subagents. Good idea?**
> 💡 Not by itself — new subagents still only get whatever the (still narrow)
> decomposition assigns them. Fix the decomposition; headcount was never the
> bug.

**6. What's the difference between a "scope gap" and a "depth gap", and why
does it matter which one you're looking at?**
> 💡 A scope gap is a whole category missing — almost always a decomposition
> bug. A depth gap is shallow coverage *within* a category you did assign —
> that one might actually be a subagent issue. Misdiagnosing one as the other
> sends you fixing the wrong layer.

**7. Bragging rights — in your own Task 6 run above, how many refinement
iterations did the *guided* decomposition actually need?**
> 💡 If it was 1: your decomposition prompt did its job well enough that the
> safety net never had to fire. If it was more than 1: you just watched the
> exact mechanism from Task 5 quietly rescue a real run — which is precisely
> what it's there for.

**8. Why keep the refinement loop at all, if a well-guided decomposition
usually gets it right on the first try?**
> 💡 Because "usually" isn't "always" — real model output varies call to
> call. The loop is the same kind of safety net as Module 1.1's
> `MAX_ITERATIONS`: not the plan, but the thing that catches it when the plan
> falls short.

---

### 🚀 Nice work.

Two modules in, and you've now built both ends of Domain 1's core idea: one
agent looping correctly on its own, and several agents cooperating without
quietly losing track of each other. Onward to
**1.3 — Subagent Invocation and Context Passing** — the implementation-level
mechanics of everything this module just covered conceptually.
